<a href="https://colab.research.google.com/github/hamshini1413/deep-learning/blob/main/14_Custom_CUDA_Accelerated_Swish_Activation_Layer_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile swish_extension.cpp
#include <torch/extension.h>
#include <vector>

// CUDA forward declaration
torch::Tensor swish_cuda_forward(torch::Tensor input, float beta);

// Python interface
torch::Tensor swish_forward(torch::Tensor input, float beta)
{
    return swish_cuda_forward(input, beta);
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)
{
    m.def("forward", &swish_forward, "Swish Forward (CUDA)");
}

Writing swish_extension.cpp


In [4]:
%%writefile swish_cuda.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void swish_kernel(
    const float* input,
    float* output,
    float beta,
    int size)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if(idx < size)
    {
        float x = input[idx];

        float sigmoid = 1.0f/(1.0f + expf(-beta*x));

        output[idx] = x * sigmoid;
    }
}

torch::Tensor swish_cuda_forward(torch::Tensor input, float beta)
{
    auto output = torch::zeros_like(input);

    int size = input.numel();

    int threads = 256;

    int blocks = (size + threads -1)/threads;

    swish_kernel<<<blocks,threads>>>(
        input.data_ptr<float>(),
        output.data_ptr<float>(),
        beta,
        size
    );

    return output;
}

Writing swish_cuda.cu


In [7]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"

from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension

# ... rest of your setup code


In [9]:
%%writefile setup.py
import os
# Set the environment variable BEFORE importing the extension tools
os.environ["CUDA_HOME"] = "/usr/local/cuda"

from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension

setup(
    name="swish_cuda",
    ext_modules=[
        CUDAExtension(
            "swish_cuda",
            [
                "swish_extension.cpp",
                "swish_cuda.cu"
            ]
        )
    ],
    cmdclass={
        "build_ext": BuildExtension
    }
)

Writing setup.py


In [23]:
import torch
import time

# -------------------------------------------------
# Device Configuration
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------------------------------
# Try importing custom CUDA extension
# -------------------------------------------------
try:
    import swish_cuda
    use_custom = True
    print("✓ Custom CUDA Extension Loaded")
except ModuleNotFoundError:
    use_custom = False
    print("⚠ swish_cuda module not found.")
    print("Using PyTorch SiLU (Swish) instead.")

# -------------------------------------------------
# Create Input Tensor
# -------------------------------------------------
x = torch.randn(1000000, device=device)

# -------------------------------------------------
# Benchmark
# -------------------------------------------------
if device == "cuda":
    torch.cuda.synchronize()

start = time.time()

if use_custom:
    for _ in range(100):
        y = swish_cuda.forward(x, 1.0)
else:
    activation = torch.nn.SiLU()
    for _ in range(100):
        y = activation(x)

if device == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nOutput Shape :", y.shape)
print("Execution Time:", elapsed, "seconds")
print("First 10 Values:\n", y[:10])

Device: cpu
⚠ swish_cuda module not found.
Using PyTorch SiLU (Swish) instead.

Output Shape : torch.Size([1000000])
Execution Time: 0.17697763442993164 seconds
First 10 Values:
 tensor([ 0.4322, -0.2770,  0.9215,  0.8624, -0.1678, -0.1318,  0.1085, -0.2419,
        -0.2606, -0.2552])


In [25]:
import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Try loading custom CUDA extension
try:
    import swish_cuda
    use_custom = True
    print("Using Custom CUDA Swish")
except ModuleNotFoundError:
    use_custom = False
    print("Custom CUDA extension not found.")
    print("Using PyTorch SiLU instead.")

# Large input tensor
x = torch.randn(10_000_000, device=device)

if device == "cuda":
    torch.cuda.synchronize()

start = time.time()

if use_custom:
    for _ in range(100):
        y = swish_cuda.forward(x, 1.0)
else:
    silu = torch.nn.SiLU()
    for _ in range(100):
        y = silu(x)

if device == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start

print(f"\nExecution Time: {elapsed:.4f} seconds")
print("Output Shape:", y.shape)

Device: cpu
Custom CUDA extension not found.
Using PyTorch SiLU instead.

Execution Time: 2.4598 seconds
Output Shape: torch.Size([10000000])
